In [1]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

In [2]:
import os

print("Current Directory Before:")
print(os.getcwd())

os.chdir("../")

print("\nCurrent Directory After:")
print(os.getcwd())

Current Directory Before:
/Users/venkatsaiganeshtadiboina/nlp_text_summarization_app/research

Current Directory After:
/Users/venkatsaiganeshtadiboina/nlp_text_summarization_app


In [3]:
import os
import torch

from dataclasses import dataclass
from pathlib import Path

from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    AutoModelForSeq2SeqLM,
    AutoTokenizer
)

from datasets import load_from_disk

from textSummarizer.constants import *
from textSummarizer.utils.common import (
    read_yaml,
    create_directories
)

/Users/venkatsaiganeshtadiboina/nlp_text_summarization_app/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path

    num_train_epochs: int
    warmup_steps: int

    per_device_train_batch_size: int

    weight_decay: float

    logging_steps: int

    evaluation_strategy: str

    eval_steps: int

    save_steps: float

    gradient_accumulation_steps: int

In [5]:
class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)

        self.params = read_yaml(params_filepath)

        create_directories(
            [self.config.artifacts_root]
        )


    def get_model_trainer_config(
        self
    ) -> ModelTrainerConfig:

        config = self.config.model_trainer

        params = self.params.TrainingArguments

        create_directories(
            [config.root_dir]
        )

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,

            data_path=config.data_path,

            model_ckpt=config.model_ckpt,

            num_train_epochs=params.num_train_epochs,

            warmup_steps=params.warmup_steps,

            per_device_train_batch_size=params.per_device_train_batch_size,

            weight_decay=params.weight_decay,

            logging_steps=params.logging_steps,

            evaluation_strategy=params.evaluation_strategy,

            eval_steps=params.eval_steps,

            save_steps=params.save_steps,

            gradient_accumulation_steps=params.gradient_accumulation_steps
        )

        return model_trainer_config

In [6]:
class ModelTrainer:

    def __init__(
        self,
        config: ModelTrainerConfig
    ):

        self.config = config


    def train(self):

        # FORCE CPU FOR STABILITY
        device = "cpu"

        print(f"Using device: {device}")


        # LOAD TOKENIZER
        tokenizer = AutoTokenizer.from_pretrained(
            self.config.model_ckpt
        )


        # LOAD MODEL
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        ).to(device)



        # DATA COLLATOR
        seq2seq_data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model_pegasus
        )


        # LOAD DATASET
        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )


        # SMALL DATASET FOR DEBUGGING
        train_data = dataset_samsum_pt["train"].select(
            range(2)
        )

        eval_data = dataset_samsum_pt["validation"].select(
            range(2)
        )


        # TRAINING ARGUMENTS
        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,

            num_train_epochs=1,

            warmup_steps=10,

            per_device_train_batch_size=1,

            weight_decay=0.01,

            logging_steps=1,

            save_steps=500,

            gradient_accumulation_steps=1,

            report_to="none"
        )


        # TRAINER
        trainer = Trainer(
            model=model_pegasus,

            args=trainer_args,

            data_collator=seq2seq_data_collator,

            train_dataset=train_data,

            eval_dataset=eval_data
        )


        # START TRAINING
        trainer.train()


        # SAVE MODEL
        model_pegasus.save_pretrained(
            os.path.join(
                self.config.root_dir,
                "pegasus-samsum-model"
            )
        )


        # SAVE TOKENIZER
        tokenizer.save_pretrained(
            os.path.join(
                self.config.root_dir,
                "tokenizer"
            )
        )


        print("\nTraining completed successfully.")

In [7]:
try:

    config = ConfigurationManager()

    model_trainer_config = (
        config.get_model_trainer_config()
    )

    model_trainer = ModelTrainer(
        config=model_trainer_config
    )

    model_trainer.train()

except Exception as e:
    raise e

[2026-05-25 09:30:16,140: INFO: common: yaml file: /Users/venkatsaiganeshtadiboina/nlp_text_summarization_app/config/config.yaml loaded successfully]
[2026-05-25 09:30:16,142: INFO: common: yaml file: /Users/venkatsaiganeshtadiboina/nlp_text_summarization_app/params.yaml loaded successfully]
[2026-05-25 09:30:16,142: INFO: common: created directory at: artifacts]
[2026-05-25 09:30:16,142: INFO: common: created directory at: artifacts/model_trainer]
Using device: cpu
[2026-05-25 09:30:16,708: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-05-25 09:30:16,711: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-05-25 09:30:16,772: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-05-25 09:30:17,066: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-25 09:30:17,126: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-25 09:30:17,403: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Foun

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 49115.32it/s]


[2026-05-25 09:30:21,822: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/xet-read-token/7db4e727842ad86cc5cdfeaef4d14c6fb2b46511 "HTTP/1.1 200 OK"]


[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-05-25 09:30:23,564: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-25 09:30:23,623: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"]


/Users/venkatsaiganeshtadiboina/nlp_text_summarization_app/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,2.995984
2,4.200768


Writing model shards: 100%|██████████| 1/1 [01:24<00:00, 84.91s/it]



Training completed successfully.
